# Selected track on an OME-Zarr temporal projection

Attach a CISegmentation 0.6.2 DuckDB or SQLite measurement database. Run the cells in OMERO.Analysis. The last cell asks the authenticated ZarrViewer to save a PNG in this notebook's results. The renderer itself accepts the same recipe from any source of track coordinates.

In [ ]:
import omero_analysis_notebook as oan
ctx = oan.configure(r'''{"schema":"nl.bioimaging.omero-analysis-notebook.v1","inputs":[{"id":"measurements","kind":"query","path":"input/measurements.duckdb","formats":["duckdb","sqlite","sqlite3"],"required":true}],"results":{"path":"results"},"parameters":[],"requirements":["pandas"]}''')
ctx.display_parameters()


In [ ]:
# Select one tracked object series from an attached CISegmentation 0.6.2 database.
chosen = await ctx.query("measurements", """
SELECT tr.track_id
FROM tracks tr
JOIN track_observations ob ON ob.track_id = tr.track_id
JOIN object_navigation nav ON nav.object_id = ob.object_id
WHERE nav.output_store_uuid IS NOT NULL AND nav.size_t > 1
GROUP BY tr.track_id, tr.observation_count
ORDER BY tr.observation_count DESC, tr.track_id
LIMIT 1
""")
if chosen.empty:
    raise ValueError("No time-series track with a linked OME-Zarr store was found")
track_id = int(chosen.iloc[0]["track_id"])
positions = await ctx.query("measurements", """
SELECT nav.output_store_uuid, nav.output_resource_path, nav.size_x, nav.size_y,
       nav.timepoint, nav.centroid_z_px, nav.centroid_y_px, nav.centroid_x_px,
       nav.label_set_id
FROM track_observations ob
JOIN object_navigation nav ON nav.object_id = ob.object_id
WHERE ob.track_id = $track_id
ORDER BY nav.timepoint, nav.object_id
LIMIT 257
""", {"track_id": track_id})
if len(positions) > 256:
    raise ValueError("Track has more than 256 observations; select a shorter interval")
if positions.empty:
    raise ValueError("Selected track has no observations")
identity = positions[["output_store_uuid", "output_resource_path"]].drop_duplicates()
if len(identity) != 1:
    raise ValueError("Selected track crosses OME-Zarr stores or fields")
channel_rows = await ctx.query("measurements", """
SELECT channel_index FROM label_sources
WHERE label_set_id = $label_set_id
ORDER BY CASE WHEN channel_role = 'primary' THEN 0 ELSE 1 END, channel_index
LIMIT 1
""", {"label_set_id": int(positions.iloc[-1]["label_set_id"])})
channel = int(channel_rows.iloc[0]["channel_index"]) if not channel_rows.empty else 1
print(f"Track {track_id}: {len(positions)} observations; intensity channel {channel}")


In [ ]:
# Request an eight-frame maximum projection with a bounded track trail.
import math

end_frame = int(positions.iloc[-1]["timepoint"])
start_frame = max(0, end_frame - 7)
recent = positions[positions["timepoint"] >= start_frame]
if recent.empty or recent[["centroid_x_px", "centroid_y_px"]].isna().any().any():
    raise ValueError("This track has missing pixel coordinates")
size_x = int(recent.iloc[-1]["size_x"])
size_y = int(recent.iloc[-1]["size_y"])
xs = [float(x) for x in recent["centroid_x_px"]]
ys = [float(y) for y in recent["centroid_y_px"]]
x0 = max(0, int(math.floor(min(xs) - 64)))
y0 = max(0, int(math.floor(min(ys) - 64)))
x1 = min(size_x, int(math.ceil(max(xs) + 64)))
y1 = min(size_y, int(math.ceil(max(ys) + 64)))
if x1 <= x0 or y1 <= y0 or x1 - x0 > 2048 or y1 - y0 > 2048:
    raise ValueError("Track crop exceeds the renderer limit; select fewer frames")
vectors = []
previous = None
for row in recent.itertuples():
    point = (float(row.centroid_x_px), float(row.centroid_y_px), int(row.timepoint))
    if previous is not None:
        vectors.append({"kind": "line", "x": previous[0], "y": previous[1],
                        "x2": point[0], "y2": point[1], "t": point[2], "z": 0,
                        "color": "#00E5FF", "width": 2, "trail": True,
                        "dashed": point[2] - previous[2] > 1})
    previous = point
vectors.append({"kind": "point", "x": previous[0], "y": previous[1],
                "t": end_frame, "z": 0, "color": "#FFB300", "radius": 5})
z_index = max(0, int(round(float(recent.iloc[-1]["centroid_z_px"] or 0))))
for item in vectors:
    item["z"] = z_index
result = {
    "omero_analysis_render_format": "png",
    "omero_analysis_render_recipe": {
        "storeUuid": str(recent.iloc[-1]["output_store_uuid"]),
        "filename": f"track-{track_id}-t{start_frame}-{end_frame}.png",
        "panels": [{
            "field": str(recent.iloc[-1]["output_resource_path"] or "."),
            "roi": [x0, y0, x1, y1], "sourceChannels": [channel],
            "t": end_frame, "z": z_index, "overlays": [],
            "title": f"Track {track_id}: t{start_frame}–{end_frame}",
            "timeProjection": {"method": "max", "start": start_frame, "end": end_frame},
            "vectors": {"version": 1, "items": vectors},
        }],
    },
}
print(f"Requesting ZarrViewer PNG for track {track_id}, t={start_frame}..{end_frame}, ROI={[x0,y0,x1,y1]}")
